# Synthetic Test Data Lab: linked records and failure fixtures

Test whether an application **accepts valid data and rejects specific errors**. This notebook creates linked employee and contract records, then makes separate copies with deliberate mistakes, such as an unknown employee ID, invalid dates, or a negative premium.

Each broken dataset includes its expected validation error. Export the records and expectations in a ZIP, feed them into your application's import or validation code, and compare the results. Saved profiles and a random seed let you reproduce the baseline without another API call.

The notebook validates its generated test cases and includes a small application-testing example. **It does not automatically test your application**; you must connect the data to your own code. The rules are example software requirements, not insurance standards.

**Get started:** run the cells in order, then click **Build test pack**. Offline demo needs no API key; hosted mode uses OpenAI for descriptive profiles. Neither mode needs model downloads or a GPU.

## 1. Dependencies and imports

The course environment includes these packages; the install cell also supports a fresh CPU notebook runtime.

In [ ]:
%pip install -q "openai>=1.109.1" python-dotenv "gradio>=5.47.2,<6" pandas

In [ ]:
import copy
import csv
import io
import json
import math
import os
import random
import tempfile
import zipfile
from datetime import date, timedelta
from pathlib import Path

import gradio as gr
import pandas as pd
from dotenv import find_dotenv, load_dotenv
from openai import OpenAI, OpenAIError

MODEL = "gpt-4.1-mini"
OFFLINE = "Offline demo (no API)"
HOSTED = "Hosted OpenAI API"
DEFAULT_CONTEXT = (
    "A fictional insurance company serving small businesses. "
    "Include varied departments, job titles, clients, and policy types."
)

def make_client():
    # Read existing course credentials only when hosted generation is requested.
    env_path = find_dotenv(usecwd=True)
    if env_path:
        load_dotenv(env_path, override=False)
    api_key = os.environ.get("OPENAI_API_KEY")
    if not api_key:
        try:
            from google.colab import userdata
        except ImportError:
            pass
        else:
            try:
                api_key = userdata.get("OPENAI_API_KEY")
            except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
                pass
    if not api_key:
        raise ValueError("No OPENAI_API_KEY is available. Use offline demo or connect your course API credentials.")
    return OpenAI(api_key=api_key, timeout=90.0)

## 2. Generate descriptive profiles

**Edit `DEFAULT_CONTEXT` or the Context box to change the business setting.** The schema stays employees + contracts; adding another table requires new construction and validation rules.

The model supplies only descriptive fields. Python controls fields whose correctness matters to the tests. Hosted output is parsed and checked before use; malformed output stops generation rather than silently becoming a fixture.

Offline profiles are fixed templates for learning and testing. They do not interpret the Context box.

In [ ]:
def check_profiles(profiles, employee_count, contract_count):
    if not isinstance(profiles, dict):
        raise ValueError("Profile output must be a JSON object.")
    specifications = {
        "employees": (employee_count, ("name", "department", "job_title")),
        "contracts": (contract_count, ("client_name", "policy_type", "description")),
    }
    cleaned = {}
    for table, (count, fields) in specifications.items():
        rows = profiles.get(table)
        if not isinstance(rows, list) or len(rows) != count:
            raise ValueError(f"Expected exactly {count} {table} profiles.")
        cleaned[table] = []
        for row in rows:
            if not isinstance(row, dict) or any(
                not isinstance(row.get(field), str) or not row[field].strip()
                for field in fields
            ):
                raise ValueError(f"Each {table} profile needs nonempty text fields: {fields}.")
            cleaned[table].append({field: row[field].strip() for field in fields})
    return cleaned


def generate_profiles(mode, context, employee_count, contract_count):
    if mode == OFFLINE:
        roles = [("Underwriting", "Underwriter"), ("Claims", "Claims analyst"),
                 ("Sales", "Account manager")]
        profiles = {
            "employees": [
                {"name": f"Fictional Employee {i + 1}",
                 "department": roles[i % len(roles)][0],
                 "job_title": roles[i % len(roles)][1]}
                for i in range(employee_count)
            ],
            "contracts": [
                {"client_name": f"Fictional Business {i + 1}",
                 "policy_type": ["Property", "Liability", "Vehicle"][i % 3],
                 "description": f"Demonstration coverage for fictional business {i + 1}."}
                for i in range(contract_count)
            ],
        }
    elif mode == HOSTED:
        if not context.strip():
            raise ValueError("Describe a business setting for hosted generation.")
        prompt = (
            f"Business setting: {context}\n"
            f"Return {employee_count} employee profiles and {contract_count} contract profiles. "
            'Return a JSON object with keys "employees" and "contracts", both arrays. '
            'Employee fields: "name", "department", "job_title". '
            'Contract fields: "client_name", "policy_type", "description". '
            "All fields must be nonempty strings. Keep each description under 25 words. "
            "Invent all people and businesses. Return only JSON, with no Markdown fences."
        )
        with make_client() as client:
            response = client.responses.create(
                model=MODEL,
                instructions="Create fictional descriptive profiles for software test fixtures.",
                input=prompt,
                max_output_tokens=6000,
            )
        if response.status != "completed":
            raise ValueError("The API response was incomplete. Try fewer records.")
        try:
            profiles = json.loads(response.output_text)
        except json.JSONDecodeError:
            raise ValueError("The API did not return valid JSON. Retry or use offline demo.") from None
    else:
        raise ValueError("Choose offline demo or hosted API.")
    return check_profiles(profiles, employee_count, contract_count)

## 3. Construct linked data with a local random seed

Each contract's `employee_id` must reference an employee. Dates use ISO `YYYY-MM-DD`; salaries must be between $30,000 and $250,000; premiums must be positive; coverage must be at least the premium.

The same **profiles + seed** produce the same dataset. The seed does **not** make hosted model responses reproducible. The export saves the profiles and seed so you can rebuild the dataset without another API request.

In [ ]:
def build_dataset(profiles, seed):
    rng = random.Random(seed)
    employees = []
    for i, profile in enumerate(profiles["employees"], start=1):
        employees.append({
            **profile,
            "employee_id": f"EMP-{i:03d}",
            "annual_salary_usd": rng.randint(40000, 180000),
        })
    contracts = []
    for i, profile in enumerate(profiles["contracts"], start=1):
        start = date(2025, 1, 1) + timedelta(days=rng.randint(0, 364))
        premium = rng.randint(500, 20000)
        contracts.append({
            **profile,
            "contract_id": f"CTR-{i:03d}",
            "employee_id": rng.choice(employees)["employee_id"],
            "start_date": start.isoformat(),
            "end_date": (start + timedelta(days=365)).isoformat(),
            "annual_premium_usd": premium,
            "coverage_limit_usd": premium * rng.randint(10, 100),
        })
    return {"employees": employees, "contracts": contracts}

## 4. Validate actual records

Validation runs in Python, independently of the model. Each error identifies a table, a **zero-based row index**, and a stable rule name that another test runner can compare.

The validator checks the documented schema and business rules. It cannot certify that model-generated descriptions are realistic or that a fictional name never matches a real person.

In [ ]:
def validate_dataset(dataset):
    errors = []
    def add(table, row, rule):
        errors.append({"table": table, "row": row, "rule": rule})
    def text(value):
        return isinstance(value, str) and bool(value.strip())
    def number(value):
        return type(value) in (int, float) and math.isfinite(value)
    def iso_date(value):
        if not isinstance(value, str):
            return None
        try:
            parsed = date.fromisoformat(value)
            return parsed if parsed.isoformat() == value else None
        except ValueError:
            return None

    if not isinstance(dataset, dict):
        return [{"table": "dataset", "row": None, "rule": "object_required"}]
    tables = {}
    for table in ("employees", "contracts"):
        rows = dataset.get(table)
        if not isinstance(rows, list) or not rows:
            add(table, None, "nonempty_table_required")
            rows = []
        tables[table] = rows

    employee_ids = set()
    for i, row in enumerate(tables["employees"]):
        if not isinstance(row, dict):
            add("employees", i, "object_required")
            continue
        for field in ("employee_id", "name", "department", "job_title"):
            if not text(row.get(field)):
                add("employees", i, f"{field}_required")
        identifier = row.get("employee_id")
        if text(identifier):
            if identifier in employee_ids:
                add("employees", i, "employee_id_unique")
            employee_ids.add(identifier)
        salary = row.get("annual_salary_usd")
        if not number(salary) or not 30000 <= salary <= 250000:
            add("employees", i, "salary_range")

    contract_ids = set()
    for i, row in enumerate(tables["contracts"]):
        if not isinstance(row, dict):
            add("contracts", i, "object_required")
            continue
        for field in ("contract_id", "client_name", "policy_type", "description"):
            if not text(row.get(field)):
                add("contracts", i, f"{field}_required")
        identifier = row.get("contract_id")
        if text(identifier):
            if identifier in contract_ids:
                add("contracts", i, "contract_id_unique")
            contract_ids.add(identifier)
        owner = row.get("employee_id")
        if not text(owner) or owner not in employee_ids:
            add("contracts", i, "employee_reference")
        start, end = iso_date(row.get("start_date")), iso_date(row.get("end_date"))
        if start is None or end is None:
            add("contracts", i, "date_format")
        elif end <= start:
            add("contracts", i, "date_order")
        premium, coverage = row.get("annual_premium_usd"), row.get("coverage_limit_usd")
        if not number(premium) or premium <= 0:
            add("contracts", i, "premium_positive")
        if not number(coverage) or coverage <= 0:
            add("contracts", i, "coverage_positive")
        elif number(premium) and premium > 0 and coverage < premium:
            add("contracts", i, "coverage_at_least_premium")
    return errors

## 5. Inject one fault per fixture

Each scenario starts from a deep copy of the validated baseline. Expected errors are defined **before validation**, not inferred from whatever the validator reports. A fixture is exported only when its actual errors exactly match its expected error.

For the duplicate employee case, a duplicate row is appended so existing contracts retain valid references. The other scenarios modify one field of the first row.

In [ ]:
SCENARIOS = {
    "salary_out_of_range": "Employee salary outside the permitted range",
    "duplicate_employee_id": "Duplicate employee ID",
    "unknown_employee": "Contract references an unknown employee",
    "reversed_dates": "Contract ends before it starts",
    "negative_premium": "Negative contract premium",
    "insufficient_coverage": "Coverage is smaller than the premium",
}

def create_fixture(baseline, scenario):
    data = copy.deepcopy(baseline)
    table, row = "contracts", 0
    if scenario == "salary_out_of_range":
        data["employees"][0]["annual_salary_usd"] = -1
        table, rule = "employees", "salary_range"
    elif scenario == "duplicate_employee_id":
        data["employees"].append(copy.deepcopy(data["employees"][0]))
        table, row, rule = "employees", len(data["employees"]) - 1, "employee_id_unique"
    elif scenario == "unknown_employee":
        data["contracts"][0]["employee_id"] = "EMP-NOT-FOUND"
        rule = "employee_reference"
    elif scenario == "reversed_dates":
        contract = data["contracts"][0]
        contract["end_date"] = (date.fromisoformat(contract["start_date"]) - timedelta(days=1)).isoformat()
        rule = "date_order"
    elif scenario == "negative_premium":
        data["contracts"][0]["annual_premium_usd"] = -1
        rule = "premium_positive"
    elif scenario == "insufficient_coverage":
        contract = data["contracts"][0]
        contract["coverage_limit_usd"] = contract["annual_premium_usd"] - 1
        rule = "coverage_at_least_premium"
    else:
        raise ValueError("Unknown fault scenario.")
    expected = [{"table": table, "row": row, "rule": rule}]
    actual = validate_dataset(data)
    if actual != expected:
        raise ValueError(f"Fixture {scenario} did not isolate its expected error.")
    return {"scenario": scenario, "expected_errors": expected,
            "actual_errors": actual, "dataset": data}


def build_test_pack(mode, context, employee_count, contract_count, seed, scenarios):
    if type(employee_count) is not int or not 1 <= employee_count <= 20:
        raise ValueError("Choose 1 to 20 employees.")
    if type(contract_count) is not int or not 1 <= contract_count <= 30:
        raise ValueError("Choose 1 to 30 contracts.")
    if type(seed) is not int:
        raise ValueError("The seed must be an integer.")
    if not isinstance(scenarios, list) or any(s not in SCENARIOS for s in scenarios):
        raise ValueError("Choose scenarios from the available list.")
    profiles = generate_profiles(mode, context, employee_count, contract_count)
    baseline = build_dataset(profiles, seed)
    baseline_errors = validate_dataset(baseline)
    if baseline_errors:
        raise ValueError("Baseline failed validation; no test pack was exported.")
    fixtures = [create_fixture(baseline, scenario) for scenario in dict.fromkeys(scenarios)]
    return {
        "metadata": {"schema_version": 1, "mode": mode,
                     "model": MODEL if mode == HOSTED else None,
                     "context": context if mode == HOSTED else None,
                     "seed": seed, "employee_count": employee_count,
                     "contract_count": contract_count},
        "profiles": profiles, "baseline": baseline,
        "baseline_errors": baseline_errors, "fixtures": fixtures,
    }

## 6. Export a test pack

The ZIP contains:

- `baseline.json`, `employees.csv`, and `contracts.csv`: valid linked records.
- `fixtures/*.json`: one independently broken dataset per scenario, with expected and actual errors.
- `profiles.json` and `metadata.json`: inputs for reproducible local reconstruction.
- `manifest.json`: baseline expectations and the expected error for each fixture.
- `README.txt`: how to interpret the pack.

These labels describe this notebook's rules. To test your application, feed the fixture datasets to its own validation or import code and compare the resulting accept/reject behavior.

In [ ]:
def export_test_pack(pack):
    def as_json(value):
        return json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False)
    def as_csv(rows):
        output = io.StringIO(newline="")
        writer = csv.DictWriter(output, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
        return output.getvalue()

    with tempfile.NamedTemporaryFile(suffix=".zip", prefix="synthetic_test_pack_", delete=False) as file:
        path = Path(file.name)
    try:
        with zipfile.ZipFile(path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
            archive.writestr("baseline.json", as_json(pack["baseline"]))
            archive.writestr("profiles.json", as_json(pack["profiles"]))
            archive.writestr("metadata.json", as_json(pack["metadata"]))
            for table in ("employees", "contracts"):
                archive.writestr(f"{table}.csv", as_csv(pack["baseline"][table]))
            manifest = [{"file": "baseline.json", "dataset_location": "root", "expected_errors": []}]
            for fixture in pack["fixtures"]:
                name = f"fixtures/{fixture['scenario']}.json"
                archive.writestr(name, as_json(fixture))
                manifest.append({"file": name, "dataset_location": "dataset",
                                 "expected_errors": fixture["expected_errors"]})
            archive.writestr("manifest.json", as_json(manifest))
            archive.writestr("README.txt", (
                "Synthetic Test Data Lab\n"
                "baseline.json should pass the notebook's documented rules.\n"
                "Each fixtures/*.json contains a dataset and one expected rule violation.\n"
                "Row numbers are zero-based. Manifest file paths are relative to this ZIP.\n"
                "Rebuild the baseline with build_dataset(saved_profiles, saved_metadata['seed']).\n"
                "The seed reproduces local construction, not a fresh hosted model response.\n"
                "Use your application's own validation code to test its behavior.\n"
            ))
    except Exception:
        path.unlink(missing_ok=True)
        raise
    return str(path)


def run_lab(mode, context, employee_count, contract_count, seed, scenarios):
    try:
        pack = build_test_pack(mode, context, int(employee_count), int(contract_count), int(seed), scenarios)
        download = export_test_pack(pack)
        rows = [{"scenario": "valid_baseline", "expected": "No errors", "verified": True}]
        rows.extend({"scenario": f["scenario"], "expected": f["expected_errors"][0]["rule"],
                     "verified": f["actual_errors"] == f["expected_errors"]} for f in pack["fixtures"])
        return (
            f"Baseline passed validation. Verified {len(pack['fixtures'])} isolated failure fixtures.",
            pd.DataFrame(pack["baseline"]["employees"]),
            pd.DataFrame(pack["baseline"]["contracts"]),
            pd.DataFrame(rows), download,
        )
    except OpenAIError:
        message = "API request failed. Check API credentials, model access, API credit, and network connectivity."
    except (ValueError, TypeError) as error:
        message = str(error)
    except OSError:
        message = "Could not save the test pack. Check local disk space and permissions."
    # Clear previous previews and downloads after any failure.
    return message, pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), None

## 7. Try the lab

Start with offline demo, inspect the valid tables, and download a pack. Switch to hosted mode for model-generated descriptive profiles. Only **Build test pack** triggers generation.

On Colab, Gradio may create a temporary public share link; anyone with the link can trigger requests while it is running. Run `demo.close()` when finished.

In [ ]:
with gr.Blocks(title="Synthetic Test Data Lab") as demo:
    gr.Markdown("# Synthetic Test Data Lab")
    gr.Markdown("Create linked employees and contracts, then export isolated failure fixtures with expected errors.")
    mode_input = gr.Radio(choices=[OFFLINE, HOSTED], value=OFFLINE, label="Profile source")
    context_input = gr.Textbox(value=DEFAULT_CONTEXT, lines=3, label="Business context (hosted mode only)")
    with gr.Row():
        employee_input = gr.Slider(1, 20, value=5, step=1, label="Employees")
        contract_input = gr.Slider(1, 30, value=8, step=1, label="Contracts")
        seed_input = gr.Number(value=42, precision=0, label="Local construction seed")
    scenario_input = gr.CheckboxGroup(
        choices=[(label, key) for key, label in SCENARIOS.items()],
        value=list(SCENARIOS), label="Failure fixtures to include",
    )
    build_button = gr.Button("Build test pack", variant="primary")
    status_output = gr.Textbox(label="Validation status", interactive=False)
    with gr.Tab("Valid employees"):
        employees_output = gr.Dataframe(interactive=False)
    with gr.Tab("Valid contracts"):
        contracts_output = gr.Dataframe(interactive=False)
    with gr.Tab("Fixture checks"):
        checks_output = gr.Dataframe(interactive=False)
    download_output = gr.File(label="Download test pack ZIP")
    build_button.click(
        fn=run_lab,
        inputs=[mode_input, context_input, employee_input, contract_input, seed_input, scenario_input],
        outputs=[status_output, employees_output, contracts_output, checks_output, download_output],
        concurrency_limit=1,
    )

demo.launch()

## 8. Download your generated files

After clicking **Build test pack**, wait for a successful validation status. Scroll to **Download test pack ZIP**, below the result tabs, and click the generated `.zip` filename or its download icon. Your browser saves the ZIP to your computer (usually Downloads, depending on your browser settings).

Open or unzip that file to find:

| File | What it contains |
| --- | --- |
| `employees.csv` | The valid employee records |
| `contracts.csv` | The valid contract records |
| `baseline.json` | Both valid tables together |
| `fixtures/unknown_employee.json` | The broken dataset and error expectations, if that scenario was selected |
| `manifest.json` | Expected validation results for the baseline and selected fixtures |

Each fixture file wraps the actual records inside its **`dataset`** field. Its `expected_errors` field tells you what the notebook's rules say should be wrong. The CSV files contain only the valid baseline; deliberately broken records are in `fixtures/`.

If no ZIP appears, check the validation status for an error and rebuild. Download results you want to keep before ending the Colab session. Building another pack updates the download field to the new pack.

The next section also shows a code-only example and an optional Colab download cell.

## 9. Test an application's contract import

Imagine an application that imports contracts into a database. Its rule is: **every contract must reference an employee who exists**.

The small application below uses a Python list instead of a real database. Its `import_contracts()` function implements its own employee-reference check; it does **not** call the generator's `validate_dataset()` function.

Run the next cell after the setup and generator function cells. It makes a fresh, separate offline pack with 3 employees, 4 contracts, and the unknown-employee fixture. It does not use the current Gradio settings, need an API key, or call another application.

The tests check that valid data is accepted, invalid data is rejected with the right error, and a rejected batch saves no extra contracts.

In [ ]:
# A tiny application: a list stands in for its database.
# Reset it whenever this cell is rerun.
saved_contracts = []


class UnknownEmployeeError(ValueError):
    pass


def import_contracts(employees, contracts):
    """Validate the entire batch before saving any contracts."""
    employee_ids = {employee["employee_id"] for employee in employees}

    for contract in contracts:
        if contract["employee_id"] not in employee_ids:
            raise UnknownEmployeeError(
                f"Unknown employee: {contract['employee_id']}"
            )

    saved_contracts.extend(copy.deepcopy(contracts))
    return len(contracts)


# Generate inputs for the application's tests, without an API call.
example_pack = build_test_pack(
    mode=OFFLINE,
    context="",
    employee_count=3,
    contract_count=4,
    seed=42,
    scenarios=["unknown_employee"],
)

# TEST 1: Valid contracts should be accepted.
valid = example_pack["baseline"]
imported = import_contracts(valid["employees"], valid["contracts"])
assert imported == 4
assert len(saved_contracts) == 4
print("PASS: All 4 valid contracts were accepted.")

# TEST 2: The broken dataset should be rejected.
broken = example_pack["fixtures"][0]["dataset"]
count_before = len(saved_contracts)

try:
    import_contracts(broken["employees"], broken["contracts"])
except UnknownEmployeeError as error:
    assert str(error) == "Unknown employee: EMP-NOT-FOUND"
    print(f"PASS: Broken dataset rejected — {error}")
else:
    raise AssertionError("FAIL: The application accepted an unknown employee!")

# TEST 3: Rejecting the batch should not save any additional contracts.
assert len(saved_contracts) == count_before
print("PASS: The rejected batch saved no additional contracts.")

Expected output:

```text
PASS: All 4 valid contracts were accepted.
PASS: Broken dataset rejected — Unknown employee: EMP-NOT-FOUND
PASS: The rejected batch saved no additional contracts.
```

**See a test catch a bug:** temporarily remove the `for contract in contracts:` validation loop from `import_contracts()` and rerun the cell. The second test should fail with `FAIL: The application accepted an unknown employee!`. Restore the loop afterward.

For a real application, replace the toy import function with its actual import function or test API. Keep the accept/reject and no-partial-save checks appropriate to that application's requirements. Use a development environment or test database.

**Download this particular example from Colab:** uncomment the three lines below and run the cell after the application example. This exports `example_pack`, not the pack currently shown in Gradio. Colab's [`files.download()`](https://github.com/googlecolab/colabtools/blob/main/google/colab/files.py) transfers the ZIP from the runtime to your computer.

In [ ]:
# Optional: uncomment all three lines to download the application example in Colab.
# example_zip = export_test_pack(example_pack)
# from google.colab import files
# files.download(example_zip)

**Use a ZIP you already generated instead:** upload that ZIP to your Colab runtime, then use this optional snippet to load its valid and broken datasets. Update the path to match your uploaded file. Select the unknown-employee scenario when generating the ZIP so that the fixture exists.

To test these loaded records, use them as `valid` and `broken` in the application checks above instead of generating a new `example_pack`. For this dataset, replace the hardcoded count `4` with `len(valid["contracts"])` and reset `saved_contracts` first.

In [ ]:
# Optional: load application-test inputs from an uploaded ZIP.
# with zipfile.ZipFile("/content/your_test_pack.zip") as archive:
#     valid = json.loads(archive.read("baseline.json"))
#     broken = json.loads(archive.read("fixtures/unknown_employee.json"))["dataset"]

## 10. Reproduce a saved baseline without an API call

After extracting the ZIP, uncomment this example and point it at the extracted directory. JSON and CSV are intended for testing; review text before opening generated CSV in a spreadsheet application.

In [ ]:
# folder = Path("extracted_test_pack")
# profiles = json.loads((folder / "profiles.json").read_text())
# metadata = json.loads((folder / "metadata.json").read_text())
# baseline = json.loads((folder / "baseline.json").read_text())
# rebuilt = build_dataset(profiles, metadata["seed"])
# assert rebuilt == baseline
# assert validate_dataset(rebuilt) == []

# Stop the interface when done:
# demo.close()

## Contribution notes and attribution

This notebook grew out of an API adaptation of [Alexandre Nkurunziza's Week 3 synthetic-data exercise](https://github.com/ed-donner/llm_engineering/blob/main/community-contributions/alexandre-nkurunziza/week_three_exercise.ipynb). That exercise informed the initial prompt-and-Gradio workflow. Its authorship is retained here; this is not a claim to have invented synthetic data generation or to have audited every community submission.

The contribution developed here is the linked-record construction pipeline, executable validation rules, isolated fault scenarios with predefined expectations, reproducible local reconstruction, and downloadable test-pack format. Those features make the work assessable beyond a model-provider or UI change.

Before submitting, run hosted mode yourself, inspect a pack, and document what you learned. This notebook was developed with AI assistance. Follow the repository's contribution and disclosure requirements.

Possible extensions: a new table and its foreign-key rules, boundary-value fixtures, an adapter that exercises your own application's validator, or tests for your application's import behavior.

References: [OpenAI text generation](https://developers.openai.com/api/docs/guides/text) · [GPT-4.1 mini](https://developers.openai.com/api/docs/models/gpt-4.1-mini)